In [35]:
import pandas as pd

df = pd.read_parquet("data/prepared_recipes.parquet")

df.head()

,name,ingredients,description,steps
0,roast butternut squash with maple syrup and gi...,"[butternut squash, ghee, maple syrup, pine nut...","found in times2, made it several times since w...","[preheat the oven to 350f, peel the squash , c..."
1,blond brownies with br sugar frosting,"[granulated sugar, vanilla, butter, eggs, flou...",have not tried these yet..but want to some day...,"[heat oven to 350 --, beat sugars , butter , v..."
2,hershey s double chocolate and peanut butter c...,"[butter, vanilla, cocoa, salt, nuts, sugar, eg...",chocolate cookies with chocolate and peanut bu...,"[preheat oven to 350 degrees, in a large mixin..."
3,latte frozen yogurt,"[sugar, cornstarch, low-fat milk, instant coff...",the flavor on this is absolutely amazing! mor...,"[in a 2-quart saucepan , combine sugar , insta..."
4,cinnamon quick bread,"[vegetable oil, egg, salt, cinnamon, sugar, bu...",yummy and easy! good with a cup of tea in the...,"[filling: mix and set aside, mix flour , bakin..."


In [36]:
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
tqdm.pandas()

model = AutoModel.from_pretrained("google/bert_uncased_L-2_H-128_A-2")
tokenizer = AutoTokenizer.from_pretrained("google/bert_uncased_L-2_H-128_A-2")

In [37]:
from vicinity import Vicinity, Backend, Metric
vicinity = Vicinity.load("data/vicinity_recipe_vectors")

In [41]:
def embed(text):
	if not isinstance(text, str):
		return None
	inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
	outputs = model(**inputs)
	return outputs.pooler_output.squeeze().detach().numpy()


# query the index
query = "chocolate cake with nuts"
query_vector = embed(query)
query_vector

array([-0.99998784, -0.17685172, -0.99169207,  0.9938672 , -0.9989181 ,
        0.8851646 , -0.88773626, -0.88067204,  0.09406027,  0.16368115,
       -0.50805485, -0.00506528,  0.04142616,  0.99994594, -0.907737  ,
       -0.57786053,  0.73683125,  0.02221121, -0.9829869 ,  0.5062256 ,
        0.8041023 , -0.07053113,  0.74148357, -0.07897776, -0.9990646 ,
        0.00732709, -0.998694  ,  0.9781059 ,  0.92987925,  0.15920979,
        0.06055846, -0.24640155, -0.9970899 , -0.7925097 ,  0.97647595,
        0.99978524, -0.95266515,  0.2055002 ,  0.6372511 , -0.9973136 ,
        0.33726826,  0.92382884, -0.9991938 ,  0.97094697, -0.9989674 ,
        0.02552438, -0.9969288 ,  0.9989138 ,  0.8870529 ,  0.96025056,
        0.95661545, -0.9420564 ,  0.00209272,  0.9791525 ,  0.94510615,
        0.9955806 , -0.9823645 , -0.6200303 ,  0.9164767 ,  0.6892927 ,
        0.07624088,  0.49738443,  0.8016696 ,  0.7881798 , -0.24337912,
       -0.9998822 , -0.19248334, -0.7074551 ,  0.9581818 ,  0.97

In [56]:
results = vicinity.query(
	query_vector,
	k=3,
)

# print the results
for result in results:
	for item in result:
		print(f"Text: {item[0]}")
		# item[0] = distance to query vector
		print(f"Distance: {item[1]}")
		print(f"Sim: {1.0 - item[1]}")
		print()

Text: the world s easiest chocolate mousse
chocolate bars, milk, honey, fresh cream
Distance: 0.012099027633666992
Sim: 0.987900972366333

Text: surprisingly sweet tortilla dip
sweet onion and pepper relish, cream cheese
Distance: 0.019821465015411377
Sim: 0.9801785349845886

Text: o j  banana breakfast smoothie
ice cubes, low-fat milk, orange juice concentrate, honey, vanilla extract, banana
Distance: 0.022624611854553223
Sim: 0.9773753881454468



In [58]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-0.6B"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)


In [59]:

# prepare the model input
prompt = f"Write a recipe for a chocolate cake with nuts, using the following ingredients: {results[0][0]}"
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)

thinking content: 
content: Here’s a **simple chocolate cake with nuts** recipe using the specified ingredients:

**Ingredients:**

- Chocolate bars (approx. 0.012099028 oz per 1 cup of cake)
- Milk
- Honey
- Fresh cream

**Instructions:**

1. **Mix chocolate bars** into a bowl and stir until smooth.
2. **In a separate bowl**, mix **milk, honey, and fresh cream** until smooth.
3. **Pour the chocolate mixture into a bowl** and **dissolve the chocolate bars** into the mixture.
4. **Use a whisk to combine** the chocolate mixture and the milk, honey, and cream.
5. **Divide the mixture into a cake mold** and **fill it with butter**.
6. **Let the cake cool** in the refrigerator for at least 1 hour.
7. **Place the cake in the refrigerator** for at least 30 minutes to make the chocolate cake more moist.
8. **Remove the cake** and **place it in the oven** to bake until it is golden and firm.
9. **Let the cake cool** in the oven for at least 1 hour.
10. **Remove the cake** and **place it in the 